## Parser

In [5]:
import os
import time
import requests
from urllib.parse import quote
from typing import Any

GITHUB_TOKEN = ''
if not GITHUB_TOKEN:
    raise ValueError("Необходимо задать GITHUB_TOKEN в переменных окружения")

HEADERS = {
    "Authorization": f"token {GITHUB_TOKEN}",
    "Accept": "application/vnd.github.v3+json"
}
TARGET_LICENSES = {"mit", "cc0-1.0", None}

DATASET_DIR = "Raw_Dataset"
os.makedirs(DATASET_DIR, exist_ok=True)

SEARCH_QUERIES = [
    "topic:python language:python",
    "topic:python-projects language:python",
]
MAX_PER_PAGE = 100
MAX_PAGES = 400
REQUEST_DELAY = 0.1
MAX_FILES = 25000

def search_repositories(query, page=1) -> tuple[Any, Any]:
    url = "https://api.github.com/search/repositories"
    params = {
        "q": query,
        "per_page": MAX_PER_PAGE,
        "page": page
    }
    resp = requests.get(url, headers=HEADERS, params=params)
    resp.raise_for_status()
    data = resp.json()
    return data["items"], data["total_count"]

def get_default_branch(owner, repo) -> Any:
    url = f"https://api.github.com/repos/{owner}/{repo}"
    resp = requests.get(url, headers=HEADERS)
    resp.raise_for_status()
    return resp.json()["default_branch"]

def get_repo_tree(owner, repo, branch) -> Any:
    url = f"https://api.github.com/repos/{owner}/{repo}/git/trees/{branch}?recursive=1"
    resp = requests.get(url, headers=HEADERS)
    resp.raise_for_status()
    return resp.json()["tree"]

def get_file_content(owner, repo, file_path, branch) -> Any | str | None:
    path_encoded = quote(file_path)
    url = f"https://api.github.com/repos/{owner}/{repo}/contents/{path_encoded}?ref={branch}"
    resp = requests.get(url, headers=HEADERS)
    if resp.status_code == 403 and "rate limit" in resp.text.lower():
        print("Request limit exceeded! Waiting")
        time.sleep(60)
        return get_file_content(owner, repo, file_path, branch)
    if resp.status_code != 200:
        print(f"Unable to get file {file_path} (status {resp.status_code})")
        return None
    data = resp.json()
    if data["type"] != "file":
        return None
    download_url = data["download_url"]
    file_resp = requests.get(download_url, headers=HEADERS)
    if file_resp.status_code != 200:
        print(f"Unable to load file: {file_path}")
        return None
    return file_resp.text

def save_file(content, repo_full_name, file_path) -> None:
    safe_name = f"{repo_full_name.replace('/', '__')}__{file_path.replace('/', '_')}"
    if len(safe_name) > 250:
        safe_name = safe_name[:250] + ".py"
    filepath = os.path.join(DATASET_DIR, safe_name)
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(content)
    print(f"Saved: {safe_name}")


def main() -> None:
    page = 1
    total_downloaded = 0
    processed_repos = 0

    for search_query in SEARCH_QUERIES:
        print('\n' * 3)
        print(f"Searching for: {search_query}")
        print('\n' * 3)

        while total_downloaded < MAX_FILES:
            print(f"Loading {page}...")
            try:
                items, total_count = search_repositories(search_query, page)
            except Exception as e:
                print(f"Error while searching: {e}")
                break

            if not items:
                print("No repos on the page")
                break

            print(f"Found repos on the page: {len(items)} (sum: {total_count})")

            for repo in items:
                repo_full_name = repo["full_name"]
                license_info = repo.get("license")
                license_key = license_info["key"] if license_info else None

                if license_key not in TARGET_LICENSES:
                    continue
                try:
                    branch = get_default_branch(*repo_full_name.split('/'))
                    time.sleep(REQUEST_DELAY)

                    tree = get_repo_tree(*repo_full_name.split('/'), branch)
                    time.sleep(REQUEST_DELAY)

                    py_files = [item for item in tree if item["type"] == "blob" and item["path"].endswith(".py")]
                    if not py_files:
                        print(" No .py files")
                        continue
                    print(f"Found .py files: {len(py_files)} if repo {repo_full_name}")

                    for file_item in py_files:
                        file_path = file_item["path"]
                        content = get_file_content(*repo_full_name.split('/'), file_path, branch)
                        if content is not None:
                            save_file(content, repo_full_name, file_path)
                            total_downloaded += 1
                        time.sleep(REQUEST_DELAY)

                except Exception as e:
                    print(f"Error while parsing repo: {e}")

                processed_repos += 1

            if len(items) < MAX_PER_PAGE:
                break 
            page += 1
            time.sleep(REQUEST_DELAY * 2) 

        print()
        print(f"Done! Parsed repos: {processed_repos}, files downloaded: {total_downloaded}")

main()





Searching for: topic:python language:python




Loading 1...
Found repos on the page: 100 (sum: 419953)
Found .py files: 1381 if repo TheAlgorithms/Python


KeyboardInterrupt: 

## Data preparation

In [5]:
"""
prepare_data.py — Pre-process raw GitHub Python files for autocomplete training.

Features:
  • Deduplication by MD5 hash
  • Quality filtering (min lines, max line length, syntax check)
  • Train / val / test split with stratification by file length
  • Statistics report + histogram saved to PNG
"""

import os, re, sys, glob, json, hashlib, ast, random, argparse, shutil
from pathlib import Path
from typing import List, Tuple, Dict
from collections import Counter
from __future__ import annotations

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np


def md5(text: str) -> str:
    return hashlib.md5(text.encode("utf-8", errors="replace")).hexdigest()


def is_valid_python(text: str) -> bool:
    try:
        ast.parse(text)
        return True
    except SyntaxError:
        return False


def basic_quality(text: str,
                  min_lines: int = 10,
                  max_lines: int = 10_000,
                  max_avg_line_len: int = 200,
                  min_code_ratio: float = 0.75) -> bool:
    lines = text.splitlines()
    if len(lines) < min_lines or len(lines) > max_lines:
        return False
    avg_len = sum(len(l) for l in lines) / max(1, len(lines))
    if avg_len > max_avg_line_len:
        return False
    # ratio of non-blank, non-comment lines
    code_lines = [l for l in lines
                  if l.strip() and not l.strip().startswith("#")]
    if len(code_lines) / max(1, len(lines)) < min_code_ratio:
        return False
    return True


def clean(text: str) -> str:
    """Light normalisation: strip trailing spaces, normalise line endings."""
    lines = text.splitlines()
    lines = [l.rstrip() for l in lines]
    # remove excessively long lines (binary / generated)
    lines = [l for l in lines if len(l) <= 500]
    return "\n".join(lines)


# ─────────────────────────────────────────────────────────────
# Stats + visualisation
# ─────────────────────────────────────────────────────────────

def compute_stats(texts: List[str]) -> Dict:
    line_counts = [len(t.splitlines()) for t in texts]
    char_counts = [len(t) for t in texts]
    token_approx = [len(t.split()) for t in texts]
    return dict(n=len(texts),
                line_counts=line_counts,
                char_counts=char_counts,
                token_approx=token_approx)


def plot_stats(stats: Dict, out_path: str) -> None:
    DARK = "#0d1117"
    MID = "#161b22"
    GRID = "#21262d"
    BLUE = "#58a6ff"
    GREEN = "#3fb950"
    ORG = "#ffa657"
    TXT = "#c9d1d9"

    plt.rcParams.update({
        "axes.facecolor":  MID, "axes.edgecolor": GRID,
        "axes.labelcolor": TXT, "xtick.color": TXT,
        "ytick.color": TXT, "text.color": TXT, "grid.color": GRID,
    })

    fig = plt.figure(figsize=(16, 8), facecolor=DARK)
    gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

    def hist(ax, data, color, label, xlabel):
        # data = {item: data.count(item) for item in set(data)}
        # x = list(data.keys())
        # height = [data[elem] for elem in x]
        # ax.bar(x, height, color=color, alpha=0.85, edgecolor=GRID)
        ax.hist(data, color=color, alpha=0.85, edgecolor=GRID)
        ax.set_xlabel(xlabel, fontsize=9)
        ax.set_ylabel("Count", fontsize=9)
        # ax.set_xscale('log')
        ax.set_yscale('log')
        ax.set_title(label, fontsize=10, color=BLUE, pad=6)
        ax.grid(True, lw=0.5)
        ax.axvline(np.median(data), color="white", lw=1.2, linestyle="--",
                   label=f"median={np.median(data):.0f}")
        ax.legend(fontsize=8)

    hist(fig.add_subplot(gs[0, 0]), stats["line_counts"],  BLUE,  "Lines per file",   "Lines")
    hist(fig.add_subplot(gs[0, 1]), stats["char_counts"],  GREEN, "Chars per file",   "Characters")
    hist(fig.add_subplot(gs[0, 2]), stats["token_approx"], ORG,   "Tokens per file",  "Tokens (approx)")

    # cumulative
    for ax_pos, data, color, label in [
        (gs[1, 0], stats["line_counts"],  BLUE,  "CDF — Lines"),
        (gs[1, 1], stats["char_counts"],  GREEN, "CDF — Chars"),
        (gs[1, 2], stats["token_approx"], ORG,   "CDF — Tokens"),
    ]:
        ax = fig.add_subplot(ax_pos)
        s = sorted(data)
        ax.plot(s, np.linspace(0, 1, len(s)), color=color, lw=1.8)
        ax.set_xlabel("Value", fontsize=9)
        # ax.set_xscale('log')
        ax.set_yscale('log')
        ax.set_ylabel("Cumulative fraction", fontsize=9)
        ax.set_title(label, fontsize=10, color=BLUE, pad=6)
        ax.grid(True, lw=0.5)

    fig.suptitle(f"Dataset Statistics — {stats['n']:,} files",
                 fontsize=13, color=BLUE, y=1.02)
    plt.savefig(out_path, dpi=130, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close(fig)
    print(f"[Stats] plot saved → {out_path}")


# ─────────────────────────────────────────────────────────────
# Main pipeline
# ─────────────────────────────────────────────────────────────

def prepare(arguments: Arguments) -> None:
    print(f"[Prepare] scanning {arguments.raw_dir} …")
    paths = glob.glob(os.path.join(arguments.raw_dir, "**/*.py"), recursive=True)
    print(f"[Prepare] found {len(paths):,} .py files")
    if arguments.delete_previous:
        if not Path.exists(Path(fr"{arguments.out_dir}")):
            print(f"[Deleting] Output directory does not exist")
        else:
            to_del_paths = glob.glob(os.path.join(arguments.out_dir, "**/*.py"), recursive=True)
            print(f"[Deleting] found {len(to_del_paths):,} .py files")
            shutil.rmtree(arguments.out_dir)
            print(f"[Deleting] Previous cleaned dataset was deleted")
            
        
    
    seen_hashes: set = set()
    accepted: List[str] = []
    stats = Counter({"total": 0, "dup": 0, "quality": 0, "ok": 0})

    for fp in paths:
        stats["total"] += 1
        try:
            raw = Path(fp).read_text(errors="replace")
        except Exception:
            continue

        # dedup
        h = md5(raw)
        if h in seen_hashes:
            stats["dup"] += 1
            continue
        seen_hashes.add(h)

        # quality
        if not basic_quality(raw, arguments.min_lines, arguments.max_lines):
            stats["quality"] += 1
            continue

        accepted.append(clean(raw))
        stats["ok"] += 1

        if arguments.max_files and len(accepted) >= arguments.max_files:
            break

    print(f"[Prepare] filter summary: {dict(stats)}")
    print(f"[Prepare] kept {len(accepted):,} files")

    # stats + plot
    s = compute_stats(accepted)
    os.makedirs(arguments.out_dir, exist_ok=True)
    plot_stats(s, os.path.join(arguments.out_dir, "dataset_stats.png"))

    # split
    random.seed(arguments.seed)
    random.shuffle(accepted)
    n = len(accepted)
    n_val = max(1, int(n * arguments.val_frac))
    n_test = max(1, int(n * arguments.test_frac))
    n_train = n - n_val - n_test

    splits = {
        "train": accepted[:n_train],
        "val": accepted[n_train: n_train + n_val],
        "test": accepted[n_train + n_val:],
    }
    for name, texts in splits.items():
        split_dir = os.path.join(arguments.out_dir, name)
        os.makedirs(split_dir, exist_ok=True)
        for i, text in enumerate(texts):
            with open(os.path.join(split_dir, f"file_{i:06d}.py"), 'w', encoding='utf-8') as file:
                file.write(text)
        print(f"[Prepare] {name}: {len(texts):,} files → {split_dir}")

    # write summary json
    summary = {
        "total_raw": stats["total"],
        "kept": stats["ok"],
        "train": n_train,
        "val": n_val,
        "test": n_test,
        "median_lines": int(np.median(s["line_counts"])),
        "median_chars": int(np.median(s["char_counts"])),
    }
    with open(os.path.join(arguments.out_dir, "summary.json"), "w") as f:
        json.dump(summary, f, indent=2)
    print(f"[Prepare] summary → {arguments.out_dir}/summary.json")


class Arguments():
    def __init__(self, raw_dir: str = 'Raw_Dataset', out_dir: str = 'Clean_Dataset', 
                    min_lines: int = 10, max_lines: int = 10_000, max_files: int = 0,
                    val_frac: float = 0.05, test_frac: float = 0.05, seed: int = 42,
                    delete_previous: bool = True) -> None:
        self.raw_dir = raw_dir
        self.out_dir = out_dir
        self.min_lines = min_lines
        self.max_lines = max_lines
        self.max_files = max_files
        self.val_frac = val_frac
        self.test_frac = test_frac
        self.seed = seed
        self.delete_previous = delete_previous



prepare(Arguments(min_lines=25, max_lines=400))
# prepare(Arguments(raw_dir="Test_Dataset", out_dir="Test_Dataset_Output"))

[Prepare] scanning Raw_Dataset …
[Prepare] found 35,197 .py files
[Deleting] found 28,628 .py files
[Deleting] Previous cleaned dataset was deleted
[Prepare] filter summary: {'total': 35197, 'dup': 3482, 'quality': 19605, 'ok': 12110}
[Prepare] kept 12,110 files
[Stats] plot saved → Clean_Dataset\dataset_stats.png
[Prepare] train: 10,900 files → Clean_Dataset\train
[Prepare] val: 605 files → Clean_Dataset\val
[Prepare] test: 605 files → Clean_Dataset\test
[Prepare] summary → Clean_Dataset/summary.json


In [ ]:
## Model Training

"""
Python Code Autocomplete — Dual Model Training
  • TokenModel  : next-token prediction (word / punctuation boundary)
  • LineModel   : full-line completion
Includes live metric visualisation, best-model checkpointing, and an
interactive hand-test REPL.
"""
%tb
import os, re, json, math, time, random, argparse, glob
from datetime import datetime
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Tuple, Optional, Dict
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

import matplotlib
matplotlib.use("Agg")           # headless — saves PNG files
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.ticker import MaxNLocator


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Device] {device}")
print(f"PyTorch версия: {torch.__version__}")
print(f"CUDA доступна: {torch.cuda.is_available()}")
print(f"Версия CUDA, под которую собран PyTorch: {torch.version.cuda}")
print(f"Количество GPU: {torch.cuda.device_count()}")


# ─────────────────────────────────────────────────────────────
# 1.  TOKENISER  (character-level BPE-lite, no dependencies)
# ─────────────────────────────────────────────────────────────
SPECIAL = {"<PAD>": 0, "<UNK>": 1, "<BOS>": 2, "<EOS>": 3}


class CodeTokenizer:
    """
    Simple sub-word tokenizer tailored for Python source code.
    Splits on whitespace/punctuation, keeps indentation tokens,
    and falls back to characters for unknowns.
    """
    PUNCT = set("()[]{}.,;:=+-*/\\%<>!&|~^@#\"'`")

    def __init__(self, vocab_size: int = 8000):
        self.vocab_size = vocab_size
        self.token2id: Dict[str, int] = dict(SPECIAL)
        self.id2token: Dict[int, str] = {v: k for k, v in SPECIAL.items()}
        self.built = False

    # ── build ──────────────────────────────────────────────
    def build(self, texts: List[str], min_freq: int = 3):
        freq: Dict[str, int] = defaultdict(int)
        for t in texts:
            for tok in self._raw_split(t):
                freq[tok] += 1
        sorted_tokens = sorted(freq.items(), key=lambda x: -x[1])
        for tok, cnt in sorted_tokens:
            if cnt < min_freq:
                break
            if tok not in self.token2id and len(self.token2id) < self.vocab_size:
                idx = len(self.token2id)
                self.token2id[tok] = idx
                self.id2token[idx] = tok
        # fill remaining slots with single chars
        for c in "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_ \t\n":
            if c not in self.token2id and len(self.token2id) < self.vocab_size:
                idx = len(self.token2id)
                self.token2id[c] = idx
                self.id2token[idx] = c
        self.built = True
        print(f"[Tokenizer] vocab_size={len(self.token2id)}")

    def _raw_split(self, text: str) -> List[str]:
        tokens = []
        for line in text.splitlines(keepends=True):
            # capture leading whitespace as indent token
            stripped = line.lstrip(" \t")
            indent = line[: len(line) - len(stripped)]
            if indent:
                tokens.append(indent)
            # split remainder on punctuation / spaces
            buf = ""
            for ch in stripped:
                if ch in self.PUNCT or ch in " \t\n\r":
                    if buf:
                        tokens.append(buf)
                        buf = ""
                    if ch.strip():
                        tokens.append(ch)
                    elif ch == "\n":
                        tokens.append("\n")
                else:
                    buf += ch
            if buf:
                tokens.append(buf)
        return tokens

    def encode(self, text: str) -> List[int]:
        ids = [SPECIAL["<BOS>"]]
        for tok in self._raw_split(text):
            if tok in self.token2id:
                ids.append(self.token2id[tok])
            else:
                # char fallback
                for ch in tok:
                    ids.append(self.token2id.get(ch, SPECIAL["<UNK>"]))
        ids.append(SPECIAL["<EOS>"])
        return ids

    def decode(self, ids: List[int]) -> str:
        parts = []
        for i in ids:
            tok = self.id2token.get(i, "")
            if tok in SPECIAL:
                continue
            parts.append(tok)
        return "".join(parts)

    def save(self, path: str):
        with open(path, "w") as f:
            json.dump({"token2id": self.token2id}, f)

    @classmethod
    def load(cls, path: str) -> "CodeTokenizer":
        with open(path) as f:
            d = json.load(f)
        obj = cls()
        obj.token2id = {k: int(v) for k, v in d["token2id"].items()}
        obj.id2token = {v: k for k, v in obj.token2id.items()}
        obj.built = True
        return obj

    @property
    def pad_id(self):  return SPECIAL["<PAD>"]
    @property
    def eos_id(self):  return SPECIAL["<EOS>"]
    @property
    def bos_id(self):  return SPECIAL["<BOS>"]
    @property
    def vocab(self):   return len(self.token2id)


# ─────────────────────────────────────────────────────────────
# 2.  DATASETS
# ─────────────────────────────────────────────────────────────

def load_files(data_dir: str, max_files: int = 0) -> List[str]:
    """Load .py / .txt files from a directory tree."""
    patterns = ["**/*.py", "**/*.txt"]
    files = []
    for pat in patterns:
        files.extend(glob.glob(os.path.join(data_dir, pat), recursive=True))
    if max_files:
        files = files[:max_files]
    texts = []
    for fp in files:
        try:
            texts.append(Path(fp).read_text(errors="replace"))
        except Exception:
            pass
    print(f"[Data] loaded {len(texts)} files from {data_dir}")
    return texts


class TokenDataset(Dataset):
    """
    Sliding-window dataset for next-token prediction.
    Target at each position is the next token id.
    """
    def __init__(self, ids: List[int], ctx: int = 128):
        self.ctx = ctx
        self.data = torch.tensor(ids, dtype=torch.long)

    def __len__(self):
        return max(0, len(self.data) - self.ctx - 1)

    def __getitem__(self, i):
        x = self.data[i: i + self.ctx]
        y = self.data[i + 1: i + self.ctx + 1]
        return x, y


class LineDataset(Dataset):
    """
    One sample = (prefix_tokens, full_line_tokens).
    The model learns to predict the rest of the current line given a prefix.
    """
    def __init__(self, texts: List[str], tokenizer: CodeTokenizer,
                 max_prefix: int = 96, max_line: int = 64):
        self.samples: List[Tuple[List[int], List[int]]] = []
        print(len([line for text in texts for line in text.splitlines()]))
        for text in texts:
            for line in text.splitlines():
                line = line.strip()
                if len(line) < 10:
                    continue
                full = tokenizer.encode(line)
                if len(full) < 4:
                    continue
                split = random.randint(2, max(2, len(full) - 2))
                prefix = full[:split][-max_prefix:]
                target = full[split:][:max_line]
                target.append(tokenizer.eos_id)
                self.samples.append((prefix, target))
        print(f"[LineDataset] {len(self.samples)} samples")

    def __len__(self): return len(self.samples)

    def __getitem__(self, i):
        return self.samples[i]


def collate_line(batch, pad_id: int):
    prefixes, targets = zip(*batch)
    max_p = max(len(p) for p in prefixes)
    max_t = max(len(t) for t in targets)
    P = torch.full((len(batch), max_p), pad_id, dtype=torch.long)
    T = torch.full((len(batch), max_t), pad_id, dtype=torch.long)
    for i, (p, t) in enumerate(zip(prefixes, targets)):
        P[i, :len(p)] = torch.tensor(p)
        T[i, :len(t)] = torch.tensor(t)
    return P, T


# ─────────────────────────────────────────────────────────────
# 3.  MODELS
# ─────────────────────────────────────────────────────────────

@dataclass
class ModelCfg:
    vocab: int = 8000
    d_model: int = 256
    n_heads: int = 8
    n_layers: int = 4
    d_ff: int = 1024
    max_len: int = 256
    dropout: float = 0.1


class PositionalEncoding(nn.Module):
    def __init__(self, d: int, max_len: int = 512, dropout: float = 0.1):
        super().__init__()
        self.drop = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d, 2).float() * (-math.log(10000.0) / d))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return self.drop(x + self.pe[:, :x.size(1)])


class TokenModel(nn.Module):
    """
    Decoder-only Transformer for causal next-token prediction.
    """
    def __init__(self, cfg: ModelCfg):
        super().__init__()
        self.cfg = cfg
        self.emb   = nn.Embedding(cfg.vocab, cfg.d_model, padding_idx=0)
        self.pos   = PositionalEncoding(cfg.d_model, cfg.max_len, cfg.dropout)
        layer      = nn.TransformerEncoderLayer(
            cfg.d_model, cfg.n_heads, cfg.d_ff, cfg.dropout,
            batch_first=True, norm_first=True
        )
        self.enc   = nn.TransformerEncoder(layer, cfg.n_layers)
        self.head  = nn.Linear(cfg.d_model, cfg.vocab, bias=False)
        self.emb.weight = self.head.weight  # weight tying

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        T = x.size(1)
        mask = nn.Transformer.generate_square_subsequent_mask(T, device=x.device)
        h = self.pos(self.emb(x))
        h = self.enc(h, mask=mask, is_causal=True)
        return self.head(h)

    @torch.no_grad()
    def generate(self, prefix_ids: List[int], max_new: int,
                 temperature: float = 0.8, top_k: int = 50,
                 stop_at_word_end: bool = True,
                 tokenizer: Optional[CodeTokenizer] = None) -> List[int]:
        self.eval()
        dev   = next(self.parameters()).device
        ids   = list(prefix_ids)
        generated = []
        PUNCT_CHARS = set("()[]{}.,;:=+-*/\\%<>!&|~^@# \t\n\"'`")
        for _ in range(max_new):
            x = torch.tensor([ids[-self.cfg.max_len:]], dtype=torch.long, device=dev)
            logits = self(x)[0, -1] / temperature
            if top_k:
                topk_v, _ = torch.topk(logits, top_k)
                logits[logits < topk_v[-1]] = -float("inf")
            probs = F.softmax(logits, dim=-1)
            nxt = torch.multinomial(probs, 1).item()
            if nxt == SPECIAL["<EOS>"]:
                break
            ids.append(nxt)
            generated.append(nxt)
            if stop_at_word_end and tokenizer:
                tok = tokenizer.id2token.get(nxt, "")
                if any(c in PUNCT_CHARS for c in tok):
                    break
        return generated


class LineModel(nn.Module):
    """
    Encoder-Decoder Transformer for seq2seq line completion.
    Encoder: reads prefix.  Decoder: generates rest of line.
    """
    def __init__(self, cfg: ModelCfg):
        super().__init__()
        self.cfg = cfg
        self.enc_emb  = nn.Embedding(cfg.vocab, cfg.d_model, padding_idx=0)
        self.dec_emb  = nn.Embedding(cfg.vocab, cfg.d_model, padding_idx=0)
        self.enc_pos  = PositionalEncoding(cfg.d_model, cfg.max_len, cfg.dropout)
        self.dec_pos  = PositionalEncoding(cfg.d_model, cfg.max_len, cfg.dropout)
        self.transformer = nn.Transformer(
            cfg.d_model, cfg.n_heads, cfg.n_layers, cfg.n_layers,
            cfg.d_ff, cfg.dropout, batch_first=True, norm_first=True
        )
        self.head = nn.Linear(cfg.d_model, cfg.vocab, bias=False)
        self.dec_emb.weight = self.head.weight

    def forward(self, src: torch.Tensor, tgt: torch.Tensor,
                src_key_padding_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        T = tgt.size(1)
        causal = nn.Transformer.generate_square_subsequent_mask(T, device=src.device)
        enc_out = self.transformer.encoder(
            self.enc_pos(self.enc_emb(src)),
            src_key_padding_mask=src_key_padding_mask
        )
        dec_out = self.transformer.decoder(
            self.dec_pos(self.dec_emb(tgt)),
            enc_out,
            tgt_mask=causal,
            tgt_is_causal=True,
            memory_key_padding_mask=src_key_padding_mask
        )
        return self.head(dec_out)

    @torch.no_grad()
    def generate(self, prefix_ids: List[int], max_new: int = 64,
                 temperature: float = 0.7, top_k: int = 40,
                 tokenizer: Optional[CodeTokenizer] = None) -> List[int]:
        self.eval()
        dev = next(self.parameters()).device
        src = torch.tensor([prefix_ids], dtype=torch.long, device=dev)
        dec_ids = [SPECIAL["<BOS>"]]
        out_ids = []
        for _ in range(max_new):
            tgt = torch.tensor([dec_ids], dtype=torch.long, device=dev)
            logits = self(src, tgt)[0, -1] / temperature
            if top_k:
                topk_v, _ = torch.topk(logits, top_k)
                logits[logits < topk_v[-1]] = -float("inf")
            probs = F.softmax(logits, dim=-1)
            nxt = torch.multinomial(probs, 1).item()
            if nxt == SPECIAL["<EOS>"]:
                break
            dec_ids.append(nxt)
            out_ids.append(nxt)
        return out_ids


# ─────────────────────────────────────────────────────────────
# 4.  METRICS & VISUALISATION
# ─────────────────────────────────────────────────────────────

@dataclass
class MetricLog:
    train_loss:  List[float] = field(default_factory=list)
    val_loss:    List[float] = field(default_factory=list)
    train_ppl:   List[float] = field(default_factory=list)
    val_ppl:     List[float] = field(default_factory=list)
    lr:          List[float] = field(default_factory=list)
    token_acc:   List[float] = field(default_factory=list)   # top-1 accuracy
    grad_norm:   List[float] = field(default_factory=list)

    def append(self, **kw):
        for k, v in kw.items():
            getattr(self, k).append(v)


def plot_metrics(log: MetricLog, title: str, save_path: str):
    """
    Rich 2×3 dashboard saved to PNG.
    """
    epochs = list(range(1, len(log.train_loss) + 1))
    fig = plt.figure(figsize=(18, 10), facecolor="#0d1117")
    gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

    ACCENT = "#58a6ff"
    WARN   = "#f78166"
    OK     = "#3fb950"
    GRID   = "#21262d"
    TXT    = "#c9d1d9"

    plt.rcParams.update({
        "axes.facecolor":   "#161b22",
        "axes.edgecolor":   GRID,
        "axes.labelcolor":  TXT,
        "xtick.color":      TXT,
        "ytick.color":      TXT,
        "text.color":       TXT,
        "grid.color":       GRID,
        "grid.linewidth":   0.6,
    })

    def _ax(pos, ylabel, title_s):
        ax = fig.add_subplot(pos)
        ax.set_xlabel("Epoch", fontsize=9)
        ax.set_ylabel(ylabel, fontsize=9)
        ax.set_title(title_s, fontsize=10, color=ACCENT, pad=6)
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))
        ax.grid(True)
        return ax

    # 1 — Loss
    ax = _ax(gs[0, 0], "Loss", "Train / Val Loss")
    ax.plot(epochs, log.train_loss, color=ACCENT, lw=1.8, label="train")
    if log.val_loss:
        ax.plot(epochs, log.val_loss, color=WARN, lw=1.8, linestyle="--", label="val")
    ax.legend(fontsize=8)

    # 2 — Perplexity
    ax = _ax(gs[0, 1], "Perplexity", "Train / Val Perplexity")
    ax.plot(epochs, log.train_ppl, color=ACCENT, lw=1.8, label="train")
    if log.val_ppl:
        ax.plot(epochs, log.val_ppl, color=WARN, lw=1.8, linestyle="--", label="val")
    ax.set_yscale("log")
    ax.legend(fontsize=8)

    # 3 — Token top-1 accuracy
    ax = _ax(gs[0, 2], "Accuracy", "Top-1 Token Accuracy")
    ax.plot(epochs, log.token_acc, color=OK, lw=1.8)
    ax.set_ylim(0, 1)

    # 4 — LR schedule
    ax = _ax(gs[1, 0], "LR", "Learning Rate")
    ax.plot(epochs, log.lr, color="#d2a8ff", lw=1.5)
    ax.set_yscale("log")

    # 5 — Gradient norm
    ax = _ax(gs[1, 1], "Grad Norm", "Gradient Norm")
    ax.plot(epochs, log.grad_norm, color="#ffa657", lw=1.5)

    # 6 — Train vs Val gap (over-fit indicator)
    ax = _ax(gs[1, 2], "Δ Loss (train-val)", "Generalisation Gap")
    if log.val_loss:
        gap = [v - t for t, v in zip(log.train_loss, log.val_loss)]
        ax.fill_between(epochs, 0, gap,
                        where=[g > 0 for g in gap], color=WARN, alpha=0.35, label="overfit")
        ax.fill_between(epochs, 0, gap,
                        where=[g <= 0 for g in gap], color=OK, alpha=0.35, label="underfit")
        ax.plot(epochs, gap, color=TXT, lw=1.0)
        ax.axhline(0, color=GRID, lw=1)
        ax.legend(fontsize=8)

    fig.suptitle(title, fontsize=14, color=ACCENT, y=1.01)
    plt.savefig(save_path, dpi=130, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close(fig)
    print(f"[Plot] saved → {save_path}")


# ─────────────────────────────────────────────────────────────
# 5.  CHECKPOINTING
# ─────────────────────────────────────────────────────────────

class BestModelSaver:
    """Keeps the best N checkpoints by val loss."""
    def __init__(self, ckpt_dir: str, model_name: str, keep: int = 3):
        self.dir   = Path(ckpt_dir)
        self.dir.mkdir(parents=True, exist_ok=True)
        self.name  = model_name
        self.keep  = keep
        self.saved: List[Tuple[float, str]] = []  # (val_loss, path)

    def save(self, model: nn.Module, val_loss: float, epoch: int, extra: dict = None):
        path = str(self.dir / f"{self.name}_ep{epoch:03d}_loss{val_loss:.4f}.pt")
        payload = {
            "model_state": model.state_dict(),
            "val_loss":    val_loss,
            "epoch":       epoch,
            "cfg":         model.cfg,
        }
        if extra:
            payload.update(extra)
        torch.save(payload, path)
        self.saved.append((val_loss, path))
        self.saved.sort(key=lambda x: x[0])
        # remove worst checkpoints beyond keep
        while len(self.saved) > self.keep:
            _, old = self.saved.pop()
            try:   os.remove(old)
            except FileNotFoundError: pass
            print(f"[Saver] removed old ckpt: {old}")
        print(f"[Saver] saved ckpt: {path}  (val_loss={val_loss:.4f})")

    def best_path(self) -> Optional[str]:
        return self.saved[0][1] if self.saved else None


# ─────────────────────────────────────────────────────────────
# 6.  TRAINING LOOPS
# ─────────────────────────────────────────────────────────────

def _clip_norm(model: nn.Module, max_norm: float = 1.0) -> float:
    return nn.utils.clip_grad_norm_(model.parameters(), max_norm).item()


def train_token_model(
    model:      TokenModel,
    train_dl:   DataLoader,
    val_dl:     DataLoader,
    epochs:     int,
    lr:         float,
    device:     torch.device,
    saver:      BestModelSaver,
    log:        MetricLog,
    plot_dir:   str,
):
    one_part = len(train_dl) // 100
    print(f"Length of train_dl: {len(train_dl)}, 1% of which is {one_part}")
    opt = AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    sched = CosineAnnealingLR(opt, T_max=epochs, eta_min=lr / 20)
    crit = nn.CrossEntropyLoss(ignore_index=SPECIAL["<PAD>"])

    for ep in range(1, epochs + 1):
        # ── train ────────────────────────────────────────────
        print(f"Epoch {ep}")
        model.train()
        t_loss = t_acc = t_steps = 0
        for index, (x, y) in enumerate(train_dl):
            if index % one_part == 0:
                print(f"{index // one_part}% at {datetime.now().time()}")
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = crit(logits.view(-1, logits.size(-1)), y.view(-1))
            opt.zero_grad()
            loss.backward()
            gn = _clip_norm(model)
            opt.step()
            t_loss += loss.item()
            preds = logits.argmax(-1)
            mask = (y != SPECIAL["<PAD>"])
            t_acc += (preds[mask] == y[mask]).float().mean().item()
            t_steps += 1

        tl = t_loss / t_steps
        ta = t_acc / t_steps

        # ── val ──────────────────────────────────────────────
        model.eval()
        v_loss = v_steps = 0
        with torch.no_grad():
            for x, y in val_dl:
                x, y = x.to(device), y.to(device)
                logits = model(x)
                loss   = crit(logits.view(-1, logits.size(-1)), y.view(-1))
                v_loss  += loss.item()
                v_steps += 1
        vl = v_loss / v_steps if v_steps else tl
        sched.step()

        log.append(train_loss=tl, val_loss=vl,
                   train_ppl=math.exp(min(tl, 20)), val_ppl=math.exp(min(vl, 20)),
                   lr=opt.param_groups[0]["lr"],
                   token_acc=ta, grad_norm=gn)

        print(f"[Token ep {ep:3d}] train_loss={tl:.4f}  val_loss={vl:.4f}"
              f"  ppl={math.exp(min(vl,20)):.1f}  acc={ta:.3f}  lr={opt.param_groups[0]['lr']:.2e}")

        saver.save(model, vl, ep)
        if ep % max(1, epochs // 5) == 0 or ep == epochs:
            plot_metrics(log, f"Token Model — Epoch {ep}", f"{plot_dir}/token_metrics_ep{ep:03d}.png")

    plot_metrics(log, "Token Model — Final", f"{plot_dir}/token_metrics_final.png")


def train_line_model(
    model: LineModel,
    train_dl: DataLoader,
    val_dl: DataLoader,
    epochs: int,
    lr:  float,
    device: torch.device,
    saver: BestModelSaver,
    log: MetricLog,
    plot_dir: str,
        ):
    one_part = len(train_dl) // 100
    print(f"Length of train_dl: {len(train_dl)}, 1% of which is {one_part}")
    opt = AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    sched = CosineAnnealingLR(opt, T_max=epochs, eta_min=lr / 20)
    crit = nn.CrossEntropyLoss(ignore_index=SPECIAL["<PAD>"])

    for ep in range(1, epochs + 1):
        print(f"Epoch {ep}")
        model.train()
        t_loss = t_acc = t_steps = 0
        for index, (src, tgt) in enumerate(train_dl):
            if index % one_part == 0:
                print(f"{index // one_part}% at {datetime.now().time()}")
            src, tgt = src.to(device), tgt.to(device)
            pad_mask = (src == SPECIAL["<PAD>"])
            dec_in = tgt[:, :-1]
            dec_out = tgt[:, 1:]
            logits = model(src, dec_in, src_key_padding_mask=pad_mask)
            loss = crit(logits.reshape(-1, logits.size(-1)), dec_out.reshape(-1))
            opt.zero_grad()
            loss.backward()
            gn = _clip_norm(model)
            opt.step()
            t_loss += loss.item()
            preds  = logits.argmax(-1)
            mask  = (dec_out != SPECIAL["<PAD>"])
            t_acc += (preds[mask] == dec_out[mask]).float().mean().item()
            t_steps += 1

        tl = t_loss / t_steps
        ta = t_acc  / t_steps

        model.eval()
        v_loss = v_steps = 0
        with torch.no_grad():
            for src, tgt in val_dl:
                src, tgt = src.to(device), tgt.to(device)
                pad_mask = (src == SPECIAL["<PAD>"])
                dec_in   = tgt[:, :-1]
                dec_out  = tgt[:, 1:]
                logits   = model(src, dec_in, src_key_padding_mask=pad_mask)
                loss     = crit(logits.reshape(-1, logits.size(-1)), dec_out.reshape(-1))
                v_loss  += loss.item()
                v_steps += 1
        vl = v_loss / v_steps if v_steps else tl
        sched.step()

        log.append(train_loss=tl, val_loss=vl,
                   train_ppl=math.exp(min(tl, 20)), val_ppl=math.exp(min(vl, 20)),
                   lr=opt.param_groups[0]["lr"],
                   token_acc=ta, grad_norm=gn)

        print(f"[Line  ep {ep:3d}] train_loss={tl:.4f}  val_loss={vl:.4f}"
              f"  ppl={math.exp(min(vl,20)):.1f}  acc={ta:.3f}  lr={opt.param_groups[0]['lr']:.2e}")

        saver.save(model, vl, ep)
        if ep % max(1, epochs // 5) == 0 or ep == epochs:
            plot_metrics(log, f"Line Model — Epoch {ep}", f"{plot_dir}/line_metrics_ep{ep:03d}.png")

    plot_metrics(log, "Line Model — Final", f"{plot_dir}/line_metrics_final.png")


# ─────────────────────────────────────────────────────────────
# 7.  INTERACTIVE HAND-TEST REPL
# ─────────────────────────────────────────────────────────────

def hand_test_repl(token_model: TokenModel, line_model: LineModel,
                   tokenizer: CodeTokenizer, device: torch.device):
    """
    Interactive REPL for testing both models.
    Commands:
      :token  <code prefix>   →  next-token completion
      :line   <code prefix>   →  full line completion
      :temp   <float>         →  set temperature
      :k      <int>           →  set top-k
      :quit                   →  exit
    """
    print("\n" + "═"*60)
    print("  Python Autocomplete — Interactive Test")
    print("  Commands: :token <prefix>  |  :line <prefix>")
    print("            :temp <float>   |  :k <int>  |  :quit")
    print("═"*60 + "\n")

    temperature = 0.8
    top_k       = 50

    while True:
        try:
            raw = input(">> ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nBye!")
            break

        if not raw:
            continue

        if raw.startswith(":quit"):
            break
        elif raw.startswith(":temp"):
            try:   temperature = float(raw.split()[1])
            except: print("Usage: :temp 0.7")
            print(f"temperature = {temperature}")
            continue
        elif raw.startswith(":k"):
            try:   top_k = int(raw.split()[1])
            except: print("Usage: :k 40")
            print(f"top_k = {top_k}")
            continue
        elif raw.startswith(":token"):
            prefix = raw[6:].strip()
            ids    = tokenizer.encode(prefix)[:-1]   # drop EOS
            with torch.no_grad():
                new_ids = token_model.generate(
                    ids, max_new=20, temperature=temperature,
                    top_k=top_k, stop_at_word_end=True, tokenizer=tokenizer)
            completion = tokenizer.decode(new_ids)
            print(f"  ← token completion: {prefix}\033[32m{completion}\033[0m\n")
        elif raw.startswith(":line"):
            prefix = raw[5:].strip()
            ids    = tokenizer.encode(prefix)[:-1]
            with torch.no_grad():
                new_ids = line_model.generate(
                    ids, max_new=64, temperature=temperature,
                    top_k=top_k, tokenizer=tokenizer)
            completion = tokenizer.decode(new_ids)
            print(f"  ← line  completion: {prefix}\033[33m{completion}\033[0m\n")
        else:
            # default to line completion
            ids = tokenizer.encode(raw)[:-1]
            with torch.no_grad():
                new_ids = line_model.generate(
                    ids, max_new=64, temperature=temperature,
                    top_k=top_k, tokenizer=tokenizer)
            completion = tokenizer.decode(new_ids)
            print(f"  ← line  completion: {raw}\033[33m{completion}\033[0m\n")


def hand_usage(token_model: TokenModel, line_model: LineModel,
                   tokenizer: CodeTokenizer, device: torch.device):
    """
    Interactive REPL for testing both models.
    Commands:
      :token  <code prefix>   →  next-token completion
      :line   <code prefix>   →  full line completion
      :temp   <float>         →  set temperature
      :k      <int>           →  set top-k
      :quit                   →  exit
    """
    print("  Python Autocomplete — Usage only")
    print("  Commands: :token <prefix>  |  :line <prefix>")
    print("            :temp <float>   |  :k <int>  |  :quit")

    from keyboard import add_hotkey

    def read_file(filename: str = 'example.py') -> tuple[str, str]:
        with open(filename, 'r', encoding='utf-8') as file:
            full_text = file.read()
            last_line = full_text.splitlines()[-1].strip()
        return full_text, last_line

    temperature = 0.8
    top_k = 50

    # end_token()

    # while True:
    #     try:
    #         full_text, last_line = read_file() 
    #     except (EOFError, KeyboardInterrupt):
    #         break

    #     if not full_text:
    #         continue

    #     if raw.startswith(":quit"):
    #         break
    #     elif raw.startswith(":temp"):
    #         try:   temperature = float(raw.split()[1])
    #         except: print("Usage: :temp 0.7")
    #         print(f"temperature = {temperature}")
    #         continue
    #     elif raw.startswith(":k"):
    #         try:   top_k = int(raw.split()[1])
    #         except: print("Usage: :k 40")
    #         print(f"top_k = {top_k}")
    #         continue
    #     elif raw.startswith(":token"):
    #         prefix = raw[6:].strip()
    #         ids = tokenizer.encode(prefix)
    #         with torch.no_grad():
    #             new_ids = token_model.generate(
    #                 ids, max_new=20, temperature=temperature,
    #                 top_k=top_k, stop_at_word_end=True, tokenizer=tokenizer)
    #         completion = tokenizer.decode(new_ids)
    #         print(f"  ← token completion: {prefix}\033[32m{completion}\033[0m\n")
    #     elif raw.startswith(":line"):
    #         prefix = raw[5:].strip()
    #         ids = tokenizer.encode(prefix)
    #         with torch.no_grad():
    #             new_ids = line_model.generate(
    #                 ids, max_new=64, temperature=temperature,
    #                 top_k=top_k, tokenizer=tokenizer)
    #         completion = tokenizer.decode(new_ids)
    #         print(f"  ← line  completion: {prefix}\033[33m{completion}\033[0m\n")
    #     else:
    #         # default to token completion
    #         ids = tokenizer.encode(raw.strip())
    #         with torch.no_grad():
    #             new_ids = token_model.generate(
    #                 ids, max_new=20, temperature=temperature,
    #                 top_k=top_k, stop_at_word_end=True, tokenizer=tokenizer)
    #         completion = tokenizer.decode(new_ids)
    #         print(f"  ← token completion: {prefix}\033[32m{completion}\033[0m\n")


    # add_hotkey()


# ─────────────────────────────────────────────────────────────
# 8.  MAIN
# ─────────────────────────────────────────────────────────────
class Arguments():
    def __init__(self, data_dir: str = "Clean_Dataset", ckpt_dir: str = "checkpoints",
                    plot_dir: str = "plots", tokenizer: str = "tokenizer.json", 
                    epochs: int = 5, batch: int = 32, lr: float = 5e-4,
                    ctx: int = 128, d_model: int = 256, n_layers: int = 4,
                    n_heads: int = 8, vocab_size: int = 000, max_files: int = 0,
                    val_split: float = 0.1, seed: int = 42, for_usage: bool = False,
                    skip_token: bool = False, skip_line: bool = False, test: bool = False):
        self.data_dir = data_dir
        self.ckpt_dir = ckpt_dir
        self.plot_dir = plot_dir
        self.tokenizer = tokenizer
        self.epochs = epochs
        self.batch = batch
        self.lr = lr
        self.ctx = ctx
        self.d_model = d_model
        self.n_layers = n_layers
        self.n_heads = n_heads
        self.vocab_size = vocab_size
        self.max_files = max_files
        self.val_split = val_split
        self.seed = seed
        self.skip_token = skip_token
        self.skip_line = skip_line
        self.test = test
        self.for_usage = for_usage


def main():
    # args = Arguments(max_files=100)
    # args = Arguments(skip_token=True, max_files=100)
    # args = Arguments(max_files=100, epochs=2, skip_token=True, vocab_size=160000)
    # args = Arguments(max_files=2500, epochs=5, skip_token=True, batch=2)
    args = Arguments(test=True)
    # args = Arguments(for_usage==True)
    random.seed(args.seed)
    np.random.seed(args.seed)
    torch.manual_seed(args.seed)

    os.makedirs(args.ckpt_dir, exist_ok=True)
    os.makedirs(args.plot_dir,  exist_ok=True)

    # ── tokenizer ────────────────────────────────────────────
    if os.path.exists(args.tokenizer):
        print(f"[Tokenizer] loading {args.tokenizer}")
        tokenizer = CodeTokenizer.load(args.tokenizer)
    else:
        print("[Tokenizer] building from data …")
        texts = load_files(args.data_dir, args.max_files)
        tokenizer = CodeTokenizer(vocab_size=args.vocab_size)
        tokenizer.build(texts)
        tokenizer.save(args.tokenizer)

    cfg = ModelCfg(
        vocab=tokenizer.vocab, d_model=args.d_model,
        n_heads=args.n_heads,  n_layers=args.n_layers,
        d_ff=args.d_model * 4, max_len=args.ctx + 32,
    )

    torch.serialization.add_safe_globals([ModelCfg])

    # ── test-only mode ───────────────────────────────────────
    if args.test:
        tok_saver  = BestModelSaver(args.ckpt_dir, "token_model")
        line_saver = BestModelSaver(args.ckpt_dir, "line_model")
        tm = TokenModel(cfg).to(device)
        lm = LineModel(cfg).to(device)
        for sav, model, name in [(tok_saver, tm, "token"), (line_saver, lm, "line")]:
            # find best ckpt by scanning dir
            paths = sorted(glob.glob(str(Path(args.ckpt_dir) / f"{name}_model_*.pt")))
            if paths:
                ck = torch.load(paths[0], map_location=device)
                model.load_state_dict(ck["model_state"])
                print(f"[Loaded] {name} from {paths[0]}")
        hand_test_repl(tm, lm, tokenizer, device)
        return
    
    
    # ──usage-only mode ───────────────────────────────────────
    if args.for_usage:
        tok_saver  = BestModelSaver(args.ckpt_dir, "token_model")
        line_saver = BestModelSaver(args.ckpt_dir, "line_model")
        tm = TokenModel(cfg).to(device)
        lm = LineModel(cfg).to(device)
        for sav, model, name in [(tok_saver, tm, "token"), (line_saver, lm, "line")]:
            # find best ckpt by scanning dir
            paths = sorted(glob.glob(str(Path(args.ckpt_dir) / f"{name}_model_*.pt")))
            if paths:
                ck = torch.load(paths[0], map_location=device)
                model.load_state_dict(ck["model_state"])
                print(f"[Loaded] {name} from {paths[0]}")
        hand_test_repl(tm, lm, tokenizer, device)
        return
    


    # ── load data ────────────────────────────────────────────
    print("[Loading] Started loading")
    texts = load_files(args.data_dir, args.max_files)
    if not texts:
        print("[ERROR] no data files found. Please put .py files in --data_dir")
        return
    print("[Loading] Ended loading")

    random.shuffle(texts)
    split = max(1, int(len(texts) * (1 - args.val_split)))
    tr_txt = texts[:split]
    va_txt = texts[split:]
    
    # ── TOKEN MODEL ──────────────────────────────────────────
    if not args.skip_token:
        print("  Prepairing TOKEN model")

        # flatten all train text → single id stream
        all_ids_tr = []
        for t in tr_txt:
            all_ids_tr.extend(tokenizer.encode(t))
        all_ids_va = []
        for t in va_txt:
            all_ids_va.extend(tokenizer.encode(t))

        tr_ds = TokenDataset(all_ids_tr, args.ctx)
        va_ds = TokenDataset(all_ids_va, args.ctx)
        tr_dl = DataLoader(tr_ds, args.batch, shuffle=True,  num_workers=0, pin_memory=True)
        va_dl = DataLoader(va_ds, args.batch, shuffle=False, num_workers=0, pin_memory=True)

        tok_model = TokenModel(cfg).to(device)
        n_params  = sum(p.numel() for p in tok_model.parameters() if p.requires_grad)
        print(f"[Token Model] {n_params/1e6:.2f}M parameters")

        tok_saver = BestModelSaver(args.ckpt_dir, "token_model")
        tok_log   = MetricLog()
        print("  Training TOKEN model")
        train_token_model(tok_model, tr_dl, va_dl, args.epochs, args.lr,
                          device, tok_saver, tok_log, args.plot_dir)
    else:
        tok_model = TokenModel(cfg).to(device)
        tok_saver = BestModelSaver(args.ckpt_dir, "token_model")
        paths = sorted(glob.glob(str(Path(args.ckpt_dir) / "token_model_*.pt")))
        if paths:
            ck = torch.load(paths[0], map_location=device)
            tok_model.load_state_dict(ck["model_state"])

    # ── LINE MODEL ───────────────────────────────────────────
    if not args.skip_line:
        print("  Prepairing LINE model")

        tr_line_ds = LineDataset(tr_txt, tokenizer)
        va_line_ds = LineDataset(va_txt, tokenizer)
        collate = lambda b: collate_line(b, tokenizer.pad_id) 
        tr_line_dl = DataLoader(tr_line_ds, args.batch, shuffle=True,
                                # num_workers=0, pin_memory=True)
                                collate_fn=collate, num_workers=0, pin_memory=True)
        va_line_dl = DataLoader(va_line_ds, args.batch, shuffle=False,
                                # num_workers=0, pin_memory=True)
                                collate_fn=collate, num_workers=0, pin_memory=True)

        line_model = LineModel(cfg).to(device)
        n_params = sum(p.numel() for p in line_model.parameters() if p.requires_grad)
        print(f"[Line  Model] {n_params/1e6:.2f}M parameters")

        line_saver = BestModelSaver(args.ckpt_dir, "line_model")
        line_log = MetricLog()
        print("  Training LINE model")
        train_line_model(line_model, tr_line_dl, va_line_dl, args.epochs, args.lr,
                         device, line_saver, line_log, args.plot_dir)
    else:
        line_model = LineModel(cfg).to(device)
        paths = sorted(glob.glob(str(Path(args.ckpt_dir) / "line_model_*.pt")))
        if paths:
            ck = torch.load(paths[0], map_location=device)
            line_model.load_state_dict(ck["model_state"])

    # ── interactive test ─────────────────────────────────────
    hand_test_repl(tok_model, line_model, tokenizer, device)


main()

RuntimeError: The size of tensor a (411) must match the size of tensor b (160) at non-singleton dimension 1

[Device] cuda
PyTorch версия: 2.6.0+cu124
CUDA доступна: True
Версия CUDA, под которую собран PyTorch: 12.4
Количество GPU: 1
[Tokenizer] loading tokenizer.json
[Loaded] token from checkpoints\token_model_ep001_loss1.2323.pt
[Loaded] line from checkpoints\line_model_ep001_loss2.7077.pt

════════════════════════════════════════════════════════════
  Python Autocomplete — Interactive Test
  Commands: :token <prefix>  |  :line <prefix>
            :temp <float>   |  :k <int>  |  :quit
════════════════════════════════════════════════════════════



>>  tok_saver  = BestModelSaver(args.ckpt_dir, "token_


  ← line  completion: tok_saver  = BestModelSaver(args.ckpt_dir, "token_sofn



>>  tok_saver  = BestMod


  ← line  completion: tok_saver  = BestModhAstomyblaCa_oibnt_oomeemlecohgnpscse



>>  :token tok_saver  = BestMod


  ← token completion: tok_saver  = BestModel



## Model evaluation

In [ ]:
"""
evaluate.py — Offline evaluation of trained autocomplete models.

Metrics computed:
  Token model : top-1 / top-5 accuracy, perplexity, mean reciprocal rank
  Line  model : exact-match@1, prefix-match, BLEU-4, chrF, perplexity
  Both        : latency (ms / sample)

Results are saved to JSON and a summary PNG dashboard.
"""

import os, json, time, glob, math, argparse
from pathlib import Path
from typing import List, Tuple, Optional

import torch
import torch.nn.functional as F
import numpy as np

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# import models from train.py (same directory)
import sys
sys.path.insert(0, os.path.dirname(__file__))


# ─────────────────────────────────────────────────────────────
# BLEU / chrF helpers (no NLTK dependency)
# ─────────────────────────────────────────────────────────────

def ngrams(seq: list, n: int) -> dict:
    counts = {}
    for i in range(len(seq) - n + 1):
        g = tuple(seq[i: i+n])
        counts[g] = counts.get(g, 0) + 1
    return counts


def bleu4(ref: list, hyp: list) -> float:
    if len(hyp) == 0:
        return 0.0
    bp = min(1.0, math.exp(1 - len(ref) / max(1, len(hyp))))
    score = 0.0
    for n in range(1, 5):
        r_ng = ngrams(ref, n)
        h_ng = ngrams(hyp, n)
        clip = sum(min(h_ng[g], r_ng.get(g, 0)) for g in h_ng)
        tot  = max(1, sum(h_ng.values()))
        if clip == 0:
            return 0.0
        score += math.log(clip / tot)
    return bp * math.exp(score / 4)


def chrf(ref: str, hyp: str, n: int = 6) -> float:
    def char_ngrams(s, n):
        return ngrams(list(s), n)
    scores = []
    for i in range(1, n + 1):
        r = char_ngrams(ref, i)
        h = char_ngrams(hyp, i)
        prec = sum(min(h[g], r.get(g, 0)) for g in h) / max(1, sum(h.values()))
        rec  = sum(min(r[g], h.get(g, 0)) for g in r) / max(1, sum(r.values()))
        f    = 2 * prec * rec / max(1e-9, prec + rec)
        scores.append(f)
    return float(np.mean(scores)) if scores else 0.0


# ─────────────────────────────────────────────────────────────
# Evaluation routines
# ─────────────────────────────────────────────────────────────

@torch.no_grad()
def eval_token_model(model: TokenModel, dataset: TokenDataset,
                     device: torch.device, n_samples: int = 2000):
    model.eval()
    crit = torch.nn.CrossEntropyLoss(ignore_index=SPECIAL["<PAD>"], reduction="sum")
    total_loss = total_tokens = 0
    top1_correct = top5_correct = 0
    mrr_sum = 0.0
    latencies = []

    indices = list(range(min(n_samples, len(dataset))))
    np.random.shuffle(indices)

    for idx in indices:
        x, y = dataset[idx]
        x = x.unsqueeze(0).to(device)
        y = y.unsqueeze(0).to(device)

        t0 = time.perf_counter()
        logits = model(x)
        latencies.append((time.perf_counter() - t0) * 1000)

        loss = crit(logits.view(-1, logits.size(-1)), y.view(-1))
        n_tok = (y != SPECIAL["<PAD>"]).sum().item()
        total_loss   += loss.item()
        total_tokens += n_tok

        # last-position metrics
        last_logit = logits[0, -1]
        true_id    = y[0, -1].item()
        if true_id == SPECIAL["<PAD>"]:
            continue
        sorted_ids = last_logit.argsort(descending=True).tolist()
        rank = sorted_ids.index(true_id) + 1 if true_id in sorted_ids else len(sorted_ids)
        top1_correct += (rank == 1)
        top5_correct += (rank <= 5)
        mrr_sum      += 1.0 / rank

    n = len(indices)
    return {
        "perplexity":  math.exp(min(total_loss / max(1, total_tokens), 20)),
        "top1_acc":    top1_correct / n,
        "top5_acc":    top5_correct / n,
        "mrr":         mrr_sum / n,
        "latency_ms":  float(np.mean(latencies)),
    }


@torch.no_grad()
def eval_line_model(model: LineModel, dataset: LineDataset,
                    tokenizer: CodeTokenizer, device: torch.device,
                    n_samples: int = 500):
    model.eval()
    exact = prefix20 = 0
    bleu_scores = []
    chrf_scores = []
    latencies   = []

    indices = list(range(min(n_samples, len(dataset))))
    np.random.shuffle(indices)

    for idx in indices:
        prefix_ids, target_ids = dataset[idx]
        # strip EOS from target for comparison
        target_ids = [i for i in target_ids if i != SPECIAL["<EOS>"]]

        t0 = time.perf_counter()
        hyp_ids = model.generate(prefix_ids, max_new=64,
                                 temperature=1.0, top_k=1,
                                 tokenizer=tokenizer)
        latencies.append((time.perf_counter() - t0) * 1000)

        exact    += (hyp_ids == target_ids)
        prefix20 += (hyp_ids[:20] == target_ids[:20])
        bleu_scores.append(bleu4(target_ids, hyp_ids))
        ref_str = tokenizer.decode(target_ids)
        hyp_str = tokenizer.decode(hyp_ids)
        chrf_scores.append(chrf(ref_str, hyp_str))

    n = len(indices)
    return {
        "exact_match":    exact / n,
        "prefix20_match": prefix20 / n,
        "bleu4":          float(np.mean(bleu_scores)),
        "chrf":           float(np.mean(chrf_scores)),
        "latency_ms":     float(np.mean(latencies)),
    }


# ─────────────────────────────────────────────────────────────
# Dashboard plot
# ─────────────────────────────────────────────────────────────

def plot_eval(tok_res: dict, line_res: dict, out_path: str):
    DARK  = "#0d1117"; MID = "#161b22"; GRID = "#21262d"
    BLUE  = "#58a6ff"; GREEN = "#3fb950"; ORG = "#ffa657"; TXT = "#c9d1d9"

    plt.rcParams.update({
        "axes.facecolor": MID, "axes.edgecolor": GRID,
        "axes.labelcolor": TXT, "xtick.color": TXT,
        "ytick.color": TXT, "text.color": TXT, "grid.color": GRID,
    })

    fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor=DARK)

    # Token model bar chart
    ax = axes[0]
    metrics = ["top1_acc", "top5_acc", "mrr"]
    values  = [tok_res[m] for m in metrics]
    labels  = ["Top-1 Acc", "Top-5 Acc", "MRR"]
    bars = ax.bar(labels, values, color=[BLUE, GREEN, ORG], width=0.5, zorder=3)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.01, f"{val:.3f}",
                ha="center", va="bottom", fontsize=10, color=TXT)
    ax.set_ylim(0, 1.05)
    ax.set_title(f"Token Model\nPerplexity: {tok_res['perplexity']:.1f}  "
                 f"Latency: {tok_res['latency_ms']:.1f}ms",
                 fontsize=10, color=BLUE, pad=8)
    ax.grid(True, axis="y", lw=0.5, zorder=0)

    # Line model bar chart
    ax = axes[1]
    metrics2 = ["exact_match", "prefix20_match", "bleu4", "chrf"]
    values2  = [line_res[m] for m in metrics2]
    labels2  = ["Exact\nMatch", "Prefix-20\nMatch", "BLEU-4", "chrF"]
    bars2 = ax.bar(labels2, values2, color=[BLUE, GREEN, ORG, "#d2a8ff"], width=0.5, zorder=3)
    for bar, val in zip(bars2, values2):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.01, f"{val:.3f}",
                ha="center", va="bottom", fontsize=10, color=TXT)
    ax.set_ylim(0, 1.05)
    ax.set_title(f"Line Model\nLatency: {line_res['latency_ms']:.1f}ms",
                 fontsize=10, color=BLUE, pad=8)
    ax.grid(True, axis="y", lw=0.5, zorder=0)

    fig.suptitle("Evaluation Dashboard", fontsize=14, color=BLUE)
    plt.tight_layout()
    plt.savefig(out_path, dpi=130, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close(fig)
    print(f"[Eval] plot → {out_path}")


# ─────────────────────────────────────────────────────────────
# Main
# ─────────────────────────────────────────────────────────────

def main():
    p = argparse.ArgumentParser()
    p.add_argument("--data_dir",   default="data/val")
    p.add_argument("--ckpt_dir",   default="checkpoints")
    p.add_argument("--tokenizer",  default="tokenizer.json")
    p.add_argument("--out_dir",    default="eval_results")
    p.add_argument("--n_tok",      type=int, default=2000)
    p.add_argument("--n_line",     type=int, default=500)
    p.add_argument("--ctx",        type=int, default=128)
    args = p.parse_args()

    os.makedirs(args.out_dir, exist_ok=True)
    device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer = CodeTokenizer.load(args.tokenizer)

    # ── load best checkpoints ────────────────────────────────
    def load_best(prefix: str, model_cls, cfg):
        paths = sorted(glob.glob(str(Path(args.ckpt_dir) / f"{prefix}_*.pt")))
        if not paths:
            print(f"[Eval] No {prefix} checkpoint found in {args.ckpt_dir}")
            return None
        ck = torch.load(paths[0], map_location=device)
        m  = model_cls(cfg).to(device)
        m.load_state_dict(ck["model_state"])
        print(f"[Eval] loaded {prefix} from {paths[0]}  (val_loss={ck['val_loss']:.4f})")
        return m

    cfg = ModelCfg(vocab=tokenizer.vocab, d_model=256, n_heads=8,
                   n_layers=4, d_ff=1024, max_len=args.ctx+32)

    tok_model  = load_best("token_model",  TokenModel, cfg)
    line_model = load_best("line_model",   LineModel,  cfg)

    texts = load_files(args.data_dir)
    if not texts:
        print("[Eval] no data found — run prepare_data.py first"); return

    all_ids = []
    for t in texts:
        all_ids.extend(tokenizer.encode(t))

    results = {}

    if tok_model:
        print("[Eval] evaluating Token model …")
        ds  = TokenDataset(all_ids, args.ctx)
        res = eval_token_model(tok_model, ds, device, args.n_tok)
        results["token"] = res
        print(json.dumps(res, indent=2))

    if line_model:
        print("[Eval] evaluating Line model …")
        ds  = LineDataset(texts, tokenizer)
        res = eval_line_model(line_model, ds, tokenizer, device, args.n_line)
        results["line"] = res
        print(json.dumps(res, indent=2))

    out_json = os.path.join(args.out_dir, "eval_results.json")
    with open(out_json, "w") as f:
        json.dump(results, f, indent=2)
    print(f"[Eval] results → {out_json}")

    if tok_model and line_model:
        plot_eval(results["token"], results["line"],
                  os.path.join(args.out_dir, "eval_dashboard.png"))


main()